In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.metrics import f1_score
import numpy as np
import neat
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from scipy.special import softmax

In [2]:
print(f"Cargando datos...")
dataset_train = pd.read_json("./data/dataset_polaridad_es_train_embeddings.json", lines=True)
dataset_test = pd.read_json("./data/dataset_polaridad_es_test_embeddings.json", lines=True)


Cargando datos...


In [3]:
LEncoder = LabelEncoder()
# El texto ya está en su forma vectorial calculado por los word embeddings del archivo según el campo
# Campo = Vector de Word Embeddings de 300 dimensiones
# we_ft = Word Embeddings calculados de textos generales del español 
# we_mx = Word Embeddings calculados de textos del español de México
# we_es = Word Embeddings calculados de textos del español de España

X_trainext = dataset_train['text'].to_numpy()
# Convertir a matriz de arrays de numpy
X_es = np.vstack(dataset_train["we_es"].to_numpy())
X_mx = np.vstack(dataset_train["we_mx"].to_numpy())
X_ft = np.vstack(dataset_train["we_ft"].to_numpy())
Y_train = dataset_train['klass'].to_numpy()

X_train = np.concatenate([X_es, X_mx, X_ft], axis=1) 


In [4]:

X_test_text = dataset_test['text'].to_numpy()
# Convertir a matriz de arrays de numpy
X_es_t = np.vstack(dataset_test["we_es"].to_numpy())
X_mx_t = np.vstack(dataset_test["we_mx"].to_numpy())
X_ft_t = np.vstack(dataset_test["we_ft"].to_numpy())
X_test = np.concatenate([X_es_t, X_mx_t, X_ft_t], axis=1) 
Y_test = dataset_test['klass'].to_numpy()

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Y_train_encoded= LEncoder.fit_transform(Y_train)
Y_test_encoded= LEncoder.transform(Y_test)

In [5]:
pca = PCA(n_components=150)

X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

In [6]:
X_train, X_val, Y_train_encoded, Y_val_encoded = train_test_split(
    X_train,
    Y_train_encoded,
    test_size=0.2,
    random_state=42,
    stratify=Y_train_encoded
)

In [7]:
def eval_genomes(genomes, config):

    BATCH_SIZE = 512

    indices = np.random.choice(len(X_train), BATCH_SIZE, replace=False)

    X_batch = X_train[indices]
    Y_batch = Y_train_encoded[indices]

    for genome_id, genome in genomes:

        net = neat.nn.FeedForwardNetwork.create(genome, config)

        predictions = []

        for xi in X_batch:
            output = net.activate(xi)
            pred = np.argmax(output)
            predictions.append(pred)

        fitness = f1_score(
            Y_batch,
            predictions,
            average="macro"
        )
        complexity_penalty = len(genome.connections) * 0.0005

        genome.fitness = fitness - complexity_penalty

In [8]:
def run_neat(config_path):
    # Cargar configuración
    config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                         neat.DefaultSpeciesSet, neat.DefaultStagnation,
                         config_path)

    # Crear la población
    p = neat.Population(config)

    # Agregar reporteros para ver el progreso en consola
    p.add_reporter(neat.StdOutReporter(True))
    stats = neat.StatisticsReporter()
    p.add_reporter(stats)

    # Ejecutar por N generaciones
    winner = p.run(eval_genomes, 50) 
    
    return winner, config


In [9]:

# Ejecución
winner_genome, config_used = run_neat('config-feedforward.cfg')


 ****** Running generation 0 ****** 

Population's average fitness: 0.26349 stdev: 0.03760
Best fitness: 0.37621 - size: (3, 90) - species 21 - id 131
Average adjusted fitness: 0.091
Mean genetic distance 2.506, standard deviation 0.336
Population of 550 members in 99 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    0    15      0.322    0.086     0
     2    0    11      0.376    0.099     0
     3    0     6      0.310    0.107     0
     4    0     7      0.319    0.096     0
     5    0    12      0.317    0.078     0
     6    0     8      0.350    0.116     0
     7    0    16      0.335    0.080     0
     8    0    13      0.356    0.112     0
     9    0    12      0.289    0.077     0
    10    0     4      0.332    0.110     0
    11    0    16      0.375    0.100     0
    12    0    13      0.328    0.101     0
    13    0    11      0.356    0.120     0
    14    0     6      0.275    0.051   

In [12]:
# Reconstruir la mejor red
winner_net = neat.nn.FeedForwardNetwork.create(winner_genome, config_used)

# Predecir en el conjunto de TEST
test_predictions = []

for xi in X_test:

    output = winner_net.activate(xi)

    probs = softmax(output)

    pred = np.argmax(probs)

    test_predictions.append(pred)

# Calcular métrica final para el reporte
final_score = f1_score(Y_test_encoded, test_predictions, average="macro")
print(f"F1-Score de la mejor red NEAT en Test: {final_score}")

F1-Score de la mejor red NEAT en Test: 0.4621040584482763
